# PySpark DataFrame Data Engineering Project — Education Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `students.csv`, `exam_results.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("students.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("exam_results.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select student_id and course from the first DataFrame.

In [ ]:
df1.select('student_id','course').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where marks is greater than its average.

In [ ]:
avg_value = df2.select(avg('marks').alias('avg')).first()['avg']
df2.filter(col('marks') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on marks using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('marks') >= avg('marks').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in course and subject with 'Unknown'.

In [ ]:
df1.fillna({'course': 'Unknown'}).show(5)
df2.fillna({'subject': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using student_id and student_id.

In [ ]:
joined = df1.join(df2, df1['student_id'] == df2['student_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['student_id'] == df2['student_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average marks by subject.

In [ ]:
df2.groupBy('subject').agg(count('*').alias('records'), avg('marks').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total marks by subject.

In [ ]:
df2.groupBy('subject').agg(sum('marks').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum marks in each subject.

In [ ]:
df2.groupBy('subject').agg(max('marks').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum marks in each subject.

In [ ]:
df2.groupBy('subject').agg(min('marks').alias('min_value')).show()

## Q14. Having Condition

Show subject groups whose total marks is greater than the overall average marks.

In [ ]:
overall = df2.select(avg('marks').alias('v')).first()['v']
df2.groupBy('subject').agg(sum('marks').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by marks.

In [ ]:
df2.orderBy(desc('marks')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by marks.

In [ ]:
df2.orderBy(asc('marks')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by marks within each subject.

In [ ]:
w = Window.partitionBy('subject').orderBy(desc('marks'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of marks within each subject.

In [ ]:
w = Window.partitionBy('subject').orderBy('exam_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('marks').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's marks with the previous record in its subject.

In [ ]:
w = Window.partitionBy('subject').orderBy('exam_date')
df2.withColumn('previous_value', lag('marks').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each subject's contribution to total marks.

In [ ]:
tot = df2.select(sum('marks').alias('t')).first()['t']
df2.groupBy('subject').agg(sum('marks').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from exam_date.

In [ ]:
df2.withColumn('year', year('exam_date')).withColumn('month', month('exam_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average marks by month.

In [ ]:
df2.withColumn('month', date_format('exam_date','yyyy-MM')).groupBy('month').agg(sum('marks').alias('total'), avg('marks').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with subject as rows and year of exam_date as columns.

In [ ]:
df2.withColumn('year', year('exam_date')).groupBy('subject').pivot('year').agg(sum('marks')).show()

## Q24. Conditional KPI

Count records where marks is above the category average.

In [ ]:
w=Window.partitionBy('subject')
df2.withColumn('cat_avg',avg('marks').over(w)).filter(col('marks')>col('cat_avg')).groupBy('subject').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total marks by course.

In [ ]:
df1.join(df2, df1['student_id']==df2['student_id']).groupBy(df1['course']).agg(sum(df2['marks']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank course by total marks.

In [ ]:
r=df1.join(df2,df1['student_id']==df2['student_id']).groupBy(df1['course']).agg(sum(df2['marks']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for marks.

In [ ]:
df2.withColumn('segment',when(col('marks')<1000,'Low').when(col('marks')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative marks values.

In [ ]:
df2.filter(col('marks')<0).count()

## Q29. Final Business Report

Create a final report by subject with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('subject').agg(count('*').alias('records'),sum('marks').alias('total'),avg('marks').alias('average'),min('marks').alias('minimum'),max('marks').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project